# Gurobi(Python调用)求解VRPTW问题
作者：yrf990409

软件：VS Code, Python 3.9, Gurobi 9.5

致谢：Gurobi(中国), 运小筹微信公众号

参考：https://doi.org/10.1137/1.9780898718515.ch7; https://mp.weixin.qq.com/s/tF-ayzjpZfuZvelvItuecw


In [1]:
# 导入所需包
import gurobipy as gp
import numpy as np
import math
import copy

## 数据处理


In [ ]:
# 导入数据
np.set_printoptions(suppress=True)    # 取消numpy打印的科学计数法
data = np.loadtxt('./Data/c101.csv',  # 相对路径下的csv文件，可替换成你的数据
                  dtype=None,         # 数据类型默认
                  encoding='UTF-8',   # 注意此文件为UTF-8格式且取消BOM
                  delimiter=',')      # 分隔符

'''
关于Slomon数据的每列数据的定义,可查看下列代码
也可以访问 https://www.sintef.no/projectweb/top/vrptw/100-customers/
下载原始txt数据
原始数据仅转换成CSV格式,并未增删,因此需要进一步处理
'''
# 数据提取，处理
# 数据切片
x_coord  = np.append(data[:,1],data[0,1])   # 横坐标
y_coord  = np.append(data[:,2],data[0,2])   # 纵坐标
demands  = np.append(data[:,3],data[0,3])   # 需求
ready_t  = np.append(data[:,4],data[0,4])   # 左时间窗
due_t    = np.append(data[:,5],data[0,5])   # 右时间窗
serve_t  = np.append(data[:,6],data[0,6])   # 服务时长

# 车辆数据
# 车辆数据的大小请查阅Solomon原始txt文件中的定义
v_cap = 200 # solomon数据集中 1系列容量为200 2系列容量为1000
v_num = 25

# 定义集合
V = [(i) for i in range(x_coord.size)] 
N = V[1:-1]
A = [(i,j) for i in V for j in V]
K = [(k) for k in range(v_num)]

# 定义时间上下界
E = float(ready_t[0])
L = float(due_t[0])

# 定义大M
M = 10000

# 定义容量
C = v_cap

# 距离矩阵计算(字典)
# 欧式距离
c = {(i,j):
    math.sqrt((x_coord[i]-x_coord[j])**2 + 
              (y_coord[i]-y_coord[j])**2)
    for i in V
    for j in V}
# print(len(c)) # 102*102

# 行驶时间矩阵
t = copy.deepcopy(c)

# 左时间窗
a = {(i):ready_t[i] for i in V}

# 右时间窗
b = {(i):due_t[i] for i in V}

# 服务时长
s = {(i):serve_t[i] for i in V}

# 客户需求
d = {(i):demands[i] for i in N}



## 定义函数

In [3]:
def DeltaPlus(i,V):
    delta_plus = copy.deepcopy(V) # 深拷贝，否则对V直接操作
    delta_plus.remove(0) # i之后访问的点不能是出发点
    if i != 0:
        delta_plus.remove(i) # i之后访问的点不能是自身
    return delta_plus

def DeltaMinus(i,V):
    delta_minus = copy.deepcopy(V) # 深拷贝，否则对V直接操作
    if i != delta_minus[-1]:
        delta_minus.remove(i) # 到达i的点不能是自身
        del delta_minus[-1]   # 到达i的点不能是返回点
    else:
        del delta_minus[-1]   # 到达i的点不能是返回点或自身
    return delta_minus


## 建模
### 实例化模型

In [4]:
m = gp.Model()

Set parameter LicenseID to value 2568715


### 创建决策变量
模型的决策变量如下：

- $x_{ijk},\,\, \forall (i,j)\in A,k\in K$为0-1决策变量，即车辆$k\in K$经过弧$(i,j)\in A$，取值则为1，否则为0。
- $w_{ik},\,\, \forall i\in V,k\in K$表示车辆$k\in K$开始服务顾客$i\in V$的时间点。

In [5]:
# 决策变量x_ijk
x = m.addVars(
    ((i, j, k) for (i, j) in A for k in K), vtype=gp.GRB.BINARY, name="x"
)  # 0-1 名称为‘x’

# 决策变量w_ik
w = m.addVars(((i, k) for i in V for k in K), vtype=gp.GRB.CONTINUOUS, name="w")

### 目标函数
$$
{\rm{min}}\,\,\ \sum_{k\in K}\sum_{(i,j)\in A}c_{ij}x_{ijk}
\tag{1}
$$

In [6]:
m.setObjective(
    gp.quicksum(c[i, j] * x[i, j, k] for (i, j) in A for k in K), sense=gp.GRB.MINIMIZE
)

### 约束
#### 一个顾客只能被一辆车服务一次

$$
\sum_{k\in K}\,\sum_{j\in \Delta^{+}(i)}x_{ijk} = 1,\,\,\forall i \in N
\tag{2}
$$

In [7]:
m.addConstrs(
    (gp.quicksum(x[i, j, k] for k in K for j in DeltaPlus(i, V)) == 1 for i in N),
    name="VServesC",
)
print()

#### 所有车辆必须出发
$$
\sum_{j\in \Delta^+(0)}x_{0jk}= 1,\,\, \forall k \in K
\tag{3}
$$

In [8]:
m.addConstrs(
    (gp.quicksum(x[0, j, k] for j in DeltaPlus(0, V)) == 1 for k in K), name="OutBound"
)
print()

#### 流守恒约束

$$
\sum_{i\in \Delta^-(j)}x_{ijk} = \sum_{i\in \Delta^+(j)}x_{jik},\,\, \forall k\in K ,\,j\in N
\tag{4}
$$

In [9]:
m.addConstrs(
    (
        gp.quicksum(x[i, j, k] for i in DeltaMinus(j, V))
        == gp.quicksum(x[j, i, k] for i in DeltaPlus(j, V))
        for k in K
        for j in N
    ),
    name="Flow",
)
print()

#### 所有车辆必须回到配送中心

$$
\sum_{i\in \Delta^-(n+1)}x_{i,n+1,k}=1,\,\, \forall k\in K
\tag{5}
$$

In [10]:
m.addConstrs(
    (gp.quicksum(x[i, V[-1], k] for i in DeltaMinus(V[-1], V)) == 1 for k in K),
    name="Inbound"
)
print()

#### 时间关系推导

$$
x_{ijk}(w_{ik}+s_i+t_{ij}-w_{jk}) \le 0,\,\,\forall k\in K,\,(i,j)\in A
\tag{不使用}
$$

$$
w_{ik}+s_i+t_{ij}-w_{jk} \le (1-x_{ijk})M,\,\,\forall k\in K,\,(i,j)\in A
\tag{6a}
$$

In [11]:
m.addConstrs(
    (
        (w[i, k] + s[i] + t[i, j] - w[j, k] <= (1 - x[i, j, k]) * M)
        for (i, j) in A
        for k in K
    ),
    name="Time",
)
print()

#### 时间窗约束

$$
a_i\sum_{j\in \Delta^+(i)}x_{ijk} \le w_{ik} \le b_i\sum_{j\in \Delta^+(i)}x_{ijk} ,\,\, \forall k \in K,\,i\in N
\tag{7}
$$

$$
E\le w_{ik}\le L,\,\, \forall k \in K ,\, i\in \{0,n+1\}
\tag{8}
$$

In [12]:
m.addConstrs(
    (
        (a[i] * (gp.quicksum(x[i, j, k] for j in DeltaPlus(i, V))) <= w[i, k])
        for k in K
        for i in N
    ),
    name="Window1",
)
m.addConstrs(
    (
        (w[i, k] <= b[i] * (gp.quicksum(x[i, j, k] for j in DeltaPlus(i, V))))
        for k in K
        for i in N
    ),
    name="Window2",
)
m.addConstrs((E <= w[i, k] for i in [0, V[-1]] for k in K), name="TimeBound1")
m.addConstrs((w[i, k] <= L for i in [0, V[-1]] for k in K), name="TimeBound2")
print()

#### 容量约束

$$
\sum_{i\in N}d_i\sum_{j\in \Delta^+(i)}x_{ijk}\le C,\,\, \forall k \in K
\tag{9}
$$

In [13]:
m.addConstrs(
    (
        gp.quicksum(d[i] * gp.quicksum(x[i, j, k] for j in DeltaPlus(i, V)) for i in N) <= C
        for k in K
    ),
    name="Cap"
)
print()

## 求解


In [14]:
m.Params.MIPGap = 0.01
m.Params.timeLimit = 7200
m.Params.LogFile =  "SolvingLog.log"

m.optimize()
m.write('Model.lp')
m.write('Solution.sol')

print('求解完成')



Set parameter MIPGap to value 0.01
Set parameter TimeLimit to value 7200
Set parameter LogFile to value "SolvingLog.log"
Gurobi Optimizer version 12.0.1 build v12.0.1rc0 (mac64[arm] - Darwin 24.5.0 24F74)

CPU model: Apple M3
Thread count: 8 physical cores, 8 logical processors, using up to 8 threads

Non-default parameters:
TimeLimit  7200
MIPGap  0.01

Optimize a model with 267875 rows, 262650 columns and 2285350 nonzeros
Model fingerprint: 0x047006f3
Variable types: 2550 continuous, 260100 integer (260100 binary)
Coefficient statistics:
  Matrix range     [1e+00, 1e+04]
  Objective range  [1e+00, 1e+02]
  Bounds range     [1e+00, 1e+00]
  RHS range        [1e+00, 1e+04]
Presolve removed 68797 rows and 65822 columns (presolve time = 5s)...
Presolve removed 71541 rows and 68566 columns
Presolve time: 5.65s
Presolved: 196334 rows, 194084 columns, 1623548 nonzeros
Variable types: 2300 continuous, 191784 integer (191784 binary)
Deterministic concurrent LP optimizer: primal simplex, dual 

## 结果分析

In [ ]:
# 获得等于每个车的弧（未排序的）
arc_list = {k: {} for k in K}
for k in K:
    arc_list[k] = {i: j for i in V for j in V if x[i, j, k].X > 0.5}

# 输出每辆车的路径
for k in K:
    print(f"Vehicle {k}: 0", end="")
    start_node = 0

    while start_node != V[-1]:
        next_node = arc_list[k][start_node]
        print(f" -> {next_node}", end="")
        start_node = next_node
    print("")


Vehicle 0: 0 -> 81 -> 78 -> 76 -> 71 -> 70 -> 73 -> 77 -> 79 -> 80 -> 101
Vehicle 1: 0 -> 43 -> 42 -> 41 -> 40 -> 44 -> 46 -> 45 -> 48 -> 51 -> 50 -> 52 -> 49 -> 47 -> 101
Vehicle 2: 0 -> 101
Vehicle 3: 0 -> 101
Vehicle 4: 0 -> 13 -> 17 -> 18 -> 19 -> 15 -> 16 -> 14 -> 12 -> 101
Vehicle 5: 0 -> 101
Vehicle 6: 0 -> 90 -> 87 -> 86 -> 83 -> 82 -> 84 -> 85 -> 88 -> 89 -> 91 -> 101
Vehicle 7: 0 -> 20 -> 24 -> 25 -> 27 -> 29 -> 30 -> 28 -> 26 -> 23 -> 22 -> 21 -> 101
Vehicle 8: 0 -> 32 -> 33 -> 31 -> 35 -> 37 -> 38 -> 39 -> 36 -> 34 -> 101
Vehicle 9: 0 -> 101
Vehicle 10: 0 -> 67 -> 65 -> 63 -> 62 -> 74 -> 72 -> 61 -> 64 -> 68 -> 66 -> 69 -> 101
Vehicle 11: 0 -> 98 -> 96 -> 95 -> 94 -> 92 -> 93 -> 97 -> 100 -> 99 -> 101
Vehicle 12: 0 -> 57 -> 55 -> 54 -> 53 -> 56 -> 58 -> 60 -> 59 -> 101
Vehicle 13: 0 -> 101
Vehicle 14: 0 -> 101
Vehicle 15: 0 -> 101
Vehicle 16: 0 -> 101
Vehicle 17: 0 -> 101
Vehicle 18: 0 -> 101
Vehicle 19: 0 -> 5 -> 3 -> 7 -> 8 -> 10 -> 11 -> 9 -> 6 -> 4 -> 2 -> 1 -> 75 -> 10